In [1]:
import uproot
import polars as pl
from tqdm import tqdm

In [ ]:
#Save CCQE events as parquet
with uproot.open("genie_dump.root") as f:
    genie = f["genie_dump"]
    keys = genie.keys()
    loaded = {}
    for key in tqdm(keys):
        loaded[key] = genie[key].array(library="np")
    genie = pl.DataFrame(loaded)


with uproot.open('nusyst_new_sum.root') as f:
    cols = ['EventID', 'SubRun', 'Run', 'Ecalo', 'Elep_calo', 'Emu_range', 'Emu_mcs', 'Ee_calo', 'Pmu_x', 'Pmu_y', 'Pmu_z', 'Pe_x', 'Pe_y', 'Pe_z']
    reco = f['SystWeights'].arrays(cols, library='pd')
    reco = pl.from_pandas(reco)

genie = pl.concat([genie, reco], how='horizontal')

genie_ccqe = genie.filter(
    pl.col('proc') == '<QES - Weak[CC]>'
)

# genie_ccqe.write_parquet('parquet/genie_ccqe.parquet')

100%|██████████| 22/22 [00:05<00:00,  4.14it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/Users/pgranger/cernbox/systematics/nusyst_new_sum.root'

In [ ]:
nucleon_mass = 0.93956542052
def get_norm_values(df):
    nb_events = df.shape[0]
    fraction_below_10MeV = df.filter(pl.col('emiss') < 0.01).shape[0] / nb_events
    fraction_above_25MeV = df.filter(pl.col('emiss') > 0.025).shape[0] / nb_events
    average_energy_above_10MeV = df.filter(pl.col('emiss') > 0.01).select(pl.col('emiss')).mean().to_numpy()[0][0]
    Eth = 0.01*(1 + np.acos(0.8)/np.pi)
    cos_average = df.filter(pl.col('emiss') > 0.01, pl.col('emiss') < 0.025).select(avg=(np.pi*pl.col('emiss')/0.01).cos()).mean().to_numpy()[0][0]
    print(Eth)

    return fraction_below_10MeV, fraction_above_25MeV, average_energy_above_10MeV, cos_average


fraction_below_10MeV, fraction_above_25MeV, average_energy_above_10MeV, cos_average = get_norm_values(genie_ccqe.with_columns(emiss = nucleon_mass - pl.col('hitnuc_E')))

In [ ]:

# %matplotlib widget

with uproot.open('DIRT2_EmissWeights_validTree.root') as f:
    Eref_counts, Eref_bins = f['prob_e_py'].to_numpy()
    Eref_counts = Eref_counts / Eref_counts.sum()
    Pref_counts, Pref_bins = f['prob_px'].to_numpy()
    Pref_counts = Pref_counts / Pref_counts.sum()

nucleon_mass = 0.93956542052
data = genie_ccqe.with_columns(
    pmiss = (pl.col('hitnuc_Px')**2 + pl.col('hitnuc_Py')**2 + pl.col('hitnuc_Pz')**2).sqrt(),
    emiss = nucleon_mass - pl.col('hitnuc_E') ,
)

pbins = np.linspace(0, 0.4, 200)
ebins = np.linspace(0, 0.05, 200)
plt.figure()
plt.hist(data['pmiss'], Pref_bins, histtype='step', label='pmiss', weights=np.ones_like(data['pmiss'])/len(data['pmiss']));
plt.step(0.5*(Pref_bins[1:] + Pref_bins[:-1]), Pref_counts, where='mid', label='pmiss reference')
plt.xlabel('pmiss [MeV]')
plt.figure()
plt.hist(data['emiss'], Eref_bins, histtype='step', label='Emiss', weights=np.ones_like(data['emiss'])/len(data['emiss']));
plt.step(0.5*(Eref_bins[1:] + Eref_bins[:-1]), Eref_counts, where='mid', label='Emiss reference')
plt.xlabel('Emiss [MeV]')

In [ ]:
# %matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import Slider

# Assuming genie_ccqe is a polars DataFrame
def reweight_tail(E, var, fraction_below_10MeV):
    return pl.when(E.is_between(0, 0.01)).then(var).otherwise(1 + fraction_below_10MeV/(1-fraction_below_10MeV)*(1 - var))
    return pl.when(E.is_between(0, 0.01)).then(var).otherwise(1)

# Assuming genie_ccqe is a polars DataFrame
def reweight_mean(E, var, fraction_below_10MeV, Eavg):
    return pl.when(E > 0.01).then(1+ var*E).otherwise(1 - var*(1 - fraction_below_10MeV)*Eavg/fraction_below_10MeV)
    return pl.when(E > 0.01).then(1+ var*E).otherwise(1)

# Assuming genie_ccqe is a polars DataFrame
def reweight_core(E, var, fraction_below_10MeV, fraction_above_25MeV, cos_average):
    # return pl.when(
    #     E.is_between(0.01, 0.025)
    #     ).then(
    #         1 + var*(1 + (np.pi*E/0.01).cos())
    #     ).otherwise(1)
    return pl.when(E.is_between(0.01, 0.025)
        ).then(
            1 + var*(1 + (np.pi*E/0.01).cos())
        ).otherwise((1 - (1 - fraction_below_10MeV - fraction_above_25MeV)*(0.2 + var*(1 - cos_average)))/(fraction_above_25MeV + fraction_below_10MeV))

nucleon_mass = 0.93956542052
data = genie_ccqe.with_columns(
    pmiss = (pl.col('hitnuc_Px')**2 + pl.col('hitnuc_Py')**2 + pl.col('hitnuc_Pz')**2).sqrt(),
    emiss = nucleon_mass - pl.col('hitnuc_E')
).with_columns(
    weight = reweight_tail(pl.col('emiss'), 1, fraction_below_10MeV)
)
ebins = np.linspace(0, 0.05, 200)
for var in [0.5, 0.7, 0.85, 1, 1.2, 1.4]:
    weight = data.select(weight = reweight_tail(pl.col('emiss'), var, fraction_below_10MeV))
    integral = weight['weight'].sum()
    # plt.hist(data['emiss'], ebins, histtype='step', label=f'Emiss {var} ; Integral {integral:.0f}', weights=weight);
    plt.hist(data['emiss'], ebins, histtype='step', label=f'Dial {var}', weights=weight);
plt.xlabel('Emiss [GeV]')
plt.legend(fontsize=12)

plt.figure()
for var in [0, 1, 1.47]:
    weight = data.select(weight = reweight_mean(pl.col('emiss'), var, fraction_below_10MeV, average_energy_above_10MeV))
    integral = weight['weight'].sum()
    # plt.hist(data['emiss'], ebins, histtype='step', label=f'Emiss {var} ; Integral {integral:.0f}', weights=weight);
    plt.hist(data['emiss'], ebins, histtype='step', label=f'Dial {var}', weights=weight);
print("sigma_limit", fraction_below_10MeV/((1 - fraction_below_10MeV)*average_energy_above_10MeV))
plt.legend(fontsize=12)

plt.figure()
for var in [0.1, 0.2, 0.3, 0.4, 0.5, 0.68]:
    weight = data.select(weight = reweight_core(pl.col('emiss'), var, fraction_below_10MeV, fraction_above_25MeV, cos_average))
    integral = weight['weight'].sum()
    plt.hist(data['emiss'], ebins, histtype='step', label=f'Emiss {var} ; Integral {integral:.0f}', weights=weight);
    # plt.hist(data['emiss'], ebins, histtype='step', label=f'Dial {var}', weights=weight);
print("sigma limit:", (0.8+0.2*fraction_below_10MeV + 0.2*fraction_above_25MeV)/(1 - fraction_below_10MeV + fraction_above_25MeV*(1 + cos_average)))
plt.legend(fontsize=12)

# plt.hist(data['emiss'], ebins, histtype='step', label='Emiss', weights=data['weight']);

# # Initial plot
# fig, ax = plt.subplots()
# plt.subplots_adjust(bottom=0.35)  # Adjust the plot to make room for the sliders

# # Initial histogram
# pbins = np.linspace(0, 300, 50)
# ebins = np.linspace(0, 100, 50)
# counts, xedges, yedges, image = ax.hist2d(data['pmiss']*1000, data['emiss']*1000, bins=30, cmap='jet', weights=data['weight'], norm='log')
# cbar = fig.colorbar(image, ax=ax)

# # Define the slider axes
# slider_ax1 = plt.axes([0.1, 0.2, 0.8, 0.03], facecolor='lightgoldenrodyellow')
# slider_ax2 = plt.axes([0.1, 0.15, 0.8, 0.03], facecolor='lightgoldenrodyellow')
# slider_ax3 = plt.axes([0.1, 0.1, 0.8, 0.03], facecolor='lightgoldenrodyellow')

# # Create the sliders
# slider1 = Slider(slider_ax1, 'Weight Scale 1', 0.1, 2.0, valinit=1.0)
# slider2 = Slider(slider_ax2, 'Weight Scale 2', 0.1, 20, valinit=1.0)
# slider3 = Slider(slider_ax3, 'Weight Scale 3', 0.1, 2.0, valinit=1.0)

# # Update function for the slider
# def update(val, func):
#     ax.cla()  # Clear the current axes
#     # Redraw the histogram with updated weights
    
#     weights = data.select(weight = func(pl.col('emiss'), val))
#     counts, xedges, yedges, image = ax.hist2d(data['pmiss']*1000, data['emiss']*1000, bins=30, cmap='jet', weights=weights['weight'], norm='log')

#     cbar.update_normal(image)  # Update the colorbar
#     ax.set_xlabel('X-axis')
#     ax.set_ylabel('Y-axis')
#     ax.set_title(f'Histogram with Weight Scale: {val:.2f}')
#     fig.canvas.draw_idle()  # Redraw the canvas

# # Connect the update function to the slider
# slider1.on_changed(lambda val: update(val, reweight_tail))
# slider2.on_changed(lambda val: update(val, reweight_mean))
# slider3.on_changed(lambda val: update(val, reweight_core))

plt.show()
